# Calculadora de Juros Compostos

Notebook organizado em blocos:
1. **Variáveis** — parâmetros de entrada
2. **Funções** — fórmulas matemáticas
3. **Execução** — processamento e resultados numéricos
4. **Visualização** — gráficos e tabelas

---
## Bloco 1 — Variáveis / Configurações

Altere os valores abaixo conforme desejar.

In [ ]:
# =========================================================================
# PARAMETROS DE ENTRADA
# =========================================================================

CAPITAL_INICIAL = 1_000.00        # Saldo inicial (R$)
TAXA_MENSAL_PCT = 1.0            # Taxa de juros mensal (%)
APORTE_MENSAL = 500.0            # Valor depositado todo mes (R$)
META = 100_000                   # Objetivo financeiro (R$)
PRAZO_MESES = 120                # Duracao da simulacao em meses

---
## Bloco 2 — Funções

Funções reutilizáveis para os cálculos.

In [ ]:
# =========================================================================
# FUNCOES AUXILIARES
# =========================================================================

def taxa_decimal(pct: float) -> float:
    """Converte porcentagem para decimal."""
    return pct / 100


def montante_futuro(pv: float, taxa: float, n: int) -> float:
    """VF = PV * (1 + i)^n"""
    return pv * (1 + taxa) ** n


def valor_presente(fv: float, taxa: float, n: int) -> float:
    """VP = FV / (1 + i)^n"""
    return fv / (1 + taxa) ** n


def projetar_mes_a_mes(capital: float, aporte: float, taxa: float,
                        meses: int) -> list[dict]:
    """Projeta saldo, rendimento e total investido mes a mes."""
    saldo = capital
    total_investido = capital
    registros = []

    for mes in range(1, meses + 1):
        rendimento = saldo * taxa
        saldo = saldo * (1 + taxa) + aporte
        total_investido += aporte
        registros.append({
            "Mes": mes,
            "Ano": round(mes / 12, 2),
            "Aporte": aporte,
            "Rendimento": round(rendimento, 2),
            "Saldo": round(saldo, 2),
            "Total_Investido": round(total_investido, 2),
        })

    return registros


def calcular_taxas_equivalentes(taxa_mensal: float) -> dict:
    """Converte taxa mensal para diaria, semestral e anual."""
    return {
        "diaria": (1 + taxa_mensal) ** (1 / 30) - 1,
        "mensal": taxa_mensal,
        "semestral": (1 + taxa_mensal) ** 6 - 1,
        "anual": (1 + taxa_mensal) ** 12 - 1,
    }


def aporte_para_meta(alvo: float, taxa: float, n: int) -> float:
    """Calcula o aporte mensal necessario para atingir o alvo em n meses."""
    if taxa <= 0:
        return alvo / n if n > 0 else float("inf")
    return alvo * taxa / ((1 + taxa) ** n - 1)


def resumo(registros: list[dict]) -> dict:
    """Extrai indicadores resumidos da projecao."""
    if not registros:
        return {}
    ult = registros[-1]
    total_inv = ult["Total_Investido"]
    total_juros = round(ult["Saldo"] - total_inv, 2)
    pct_juros = total_juros / ult["Saldo"] * 100 if ult["Saldo"] else 0
    return {
        "saldo_final": ult["Saldo"],
        "total_investido": total_inv,
        "total_juros": total_juros,
        "pct_juros": round(pct_juros, 2),
        "meses": ult["Mes"],
        "anos": round(ult["Mes"] / 12, 1),
    }

---
## Bloco 3 — Execução

Executa os cálculos com os parâmetros do Bloco 1.

In [ ]:
# =========================================================================
# PREPARACAO
# =========================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

TAXA_DECIMAL = taxa_decimal(TAXA_MENSAL_PCT)  # Converte % para decimal

In [ ]:
# =========================================================================
# PROJECAO MES A MES
# =========================================================================

registros = projetar_mes_a_mes(CAPITAL_INICIAL, APORTE_MENSAL,
                                TAXA_DECIMAL, PRAZO_MESES)
df = pd.DataFrame(registros)
r = resumo(registros)

print("--- Projecao ---")
print(f"Saldo final: R$ {r['saldo_final']:,.2f}")
print(f"Total investido: R$ {r['total_investido']:,.2f}")
print(f"Total em juros: R$ {r['total_juros']:,.2f} ({r['pct_juros']}%)")
print(f"Periodo: {r['meses']} meses ({r['anos']} anos)")

In [ ]:
# =========================================================================
# TAXAS EQUIVALENTES
# =========================================================================

taxas = calcular_taxas_equivalentes(TAXA_DECIMAL)
print("--- Taxas Equivalentes ---")
for periodo, t in taxas.items():
    print(f"  {periodo.capitalize():>9}: {t * 100:.4f}%")

In [ ]:
# =========================================================================
# APORTE NECESSARIO PARA ATINGIR A META
# =========================================================================

for anos in [5, 10, 20, 30]:
    ap = aporte_para_meta(META, TAXA_DECIMAL, anos * 12)
    print(f"  Em {anos:>2} anos: R$ {ap:>10,.2f}/mes")

# Quanto tempo para atingir a meta com o aporte atual?
linha_meta = df[df["Saldo"] >= META].head(1)

if not linha_meta.empty:
    mes_meta = int(linha_meta.iloc[0]["Mes"])
    print(f"\nMeta de R$ {META:,.2f} atingida em {mes_meta} meses "
          f"({mes_meta // 12}a {mes_meta % 12}m)")
else:
    print(f"\nMeta de R$ {META:,.2f} NAO atingida em {PRAZO_MESES} meses.")

---
## Bloco 4 — Visualização

Gráficos da evolução patrimonial.

In [ ]:
# =========================================================================
# CONFIGURACAO DOS GRAFICOS
# =========================================================================

plt.rcParams["figure.figsize"] = (14, 6)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

fmt_reais = mticker.FuncFormatter(
    lambda x, _: f"R$ {x/1000:,.0f}k" if x >= 1000 else f"R$ {x:,.0f}"
)

In [ ]:
# =========================================================================
# GRAFICO 1 - EVOLUCAO DO PATRIMONIO
# =========================================================================

fig, ax = plt.subplots()

ax.plot(df["Mes"], df["Saldo"], label="Saldo acumulado",
        linewidth=2, color="#2196F3")
ax.plot(df["Mes"], df["Total_Investido"], label="Total aportado",
        linewidth=1.5, color="#4CAF50", linestyle="--")
ax.fill_between(df["Mes"], df["Total_Investido"], df["Saldo"],
                alpha=0.15, color="#2196F3", label="Juros compostos")

if META:
    ax.axhline(y=META, color="#FF9800", linestyle="--", alpha=0.7,
               label=f"Meta: R$ {META:,.0f}")

ax.yaxis.set_major_formatter(fmt_reais)
ax.set_xlabel("Meses")
ax.set_ylabel("Valor (R$)")
ax.set_title("Evolucao do Patrimonio")
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
# =========================================================================
# GRAFICO 2 - COMPOSICAO DO MONTANTE FINAL
# =========================================================================

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

labels = ["Total investido", "Rendimentos"]
valores = [r["total_investido"], r["total_juros"]]
cores = ["#4CAF50", "#2196F3"]

ax1.pie(valores, labels=labels, autopct="%1.1f%%",
        startangle=90, colors=cores, explode=(0, 0.05))
ax1.set_title("Composicao do Montante")

barras = ax2.bar(labels, valores, color=cores, width=0.5)
offset = max(valores) * 0.02 if max(valores) else 100
for barra, valor in zip(barras, valores):
    ax2.text(barra.get_x() + barra.get_width() / 2,
             barra.get_height() + offset,
             f"R$ {valor:,.2f}", ha="center",
             fontsize=11, fontweight="bold")
ax2.yaxis.set_major_formatter(fmt_reais)
ax2.set_title("Investido vs Juros")
plt.tight_layout()
plt.show()

In [ ]:
# =========================================================================
# TABELA - RESUMO COMPLETO
# =========================================================================

print("=" * 60)
print("RESUMO DA SIMULACAO")
print("=" * 60)
print(f"\nParametros:")
print(f"  Capital inicial:  R$ {CAPITAL_INICIAL:>10,.2f}")
print(f"  Taxa mensal:      {TAXA_MENSAL_PCT:>10.2f}%")
print(f"  Taxa anual:       {taxas['anual'] * 100:>10.2f}%")
print(f"  Aporte mensal:    R$ {APORTE_MENSAL:>10,.2f}")
print(f"  Meta:             R$ {META:>10,.2f}")
print(f"  Prazo:            {PRAZO_MESES:>10} meses ({PRAZO_MESES // 12} anos)")
print(f"\nResultado:")
print(f"  Saldo final:      R$ {r['saldo_final']:>10,.2f}")
print(f"  Total investido:  R$ {r['total_investido']:>10,.2f}")
print(f"  Total em juros:   R$ {r['total_juros']:>10,.2f} ({r['pct_juros']}%)")

In [ ]:
# =========================================================================
# TABELA MES A MES (AMOSTRA)
# =========================================================================

print("\nPrimeiras 10 linhas:")
display(df.head(10))
print(f"\nTotal de registros: {len(df)}")

In [ ]:
# =========================================================================
# SENSIBILIDADE - IMPACTO DA TAXA NO SALDO FINAL
# =========================================================================

taxas_teste = np.arange(0.25, 2.25, 0.25)
saldos_teste = []

for t in taxas_teste:
    reg = projetar_mes_a_mes(CAPITAL_INICIAL, APORTE_MENSAL,
                              taxa_decimal(t), PRAZO_MESES)
    saldos_teste.append(reg[-1]["Saldo"])

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar([f"{t:.1f}%" for t in taxas_teste], saldos_teste,
       color="#2196F3", width=0.6)
ax.yaxis.set_major_formatter(fmt_reais)
ax.set_xlabel("Taxa mensal")
ax.set_ylabel("Saldo final (R$)")
ax.set_title(f"Impacto da Taxa no Saldo Final ({PRAZO_MESES} meses)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()